# Qian — LoRA crop and resolution experiments
Use the team's strongest **LoRA r24 + extras** recipe with the required IN21K→IN1K ViT-B/16. No VPT. Train on 1,000 classes with five images/class held out; fitting images alone build fresh prototypes at each resolution. The head remains 10,000-way with original labels. Classes outside the subset start with seeded random head rows; holdout and full-data prototypes are never used to initialize them.

| Arm | Resolution | Image preparation |
|---|---:|---|
| A | 336 | Current random crop |
| B | 336 | 75% current crop + 25% whole image with padding |
| C | 384 | Current random crop |
| D | 384 | 75% current crop + 25% whole image with padding |

Every image gets **one view per visit**. Flip, color changes, normalization, optimizer, seed, and LoRA settings stay the same. Choose an **A100 GPU runtime**. This notebook trains models; the CPU advice for the inspection notebook does not apply here.

Default: **12 training minutes per arm** (48 for A–D), plus extraction, prototypes, evaluation and saving. Optional E adds 12 minutes. Those other steps take additional wall time; this is not four 144-minute full-data runs. Training timing includes image loading and augmentation. Results shortlist candidates, not final full-data scores.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/qq_lora_ablation_repo"
BRANCH = "qq_lora_ablation"  # Publish this branch before opening the launcher in Colab.
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
EXPERIMENT_NAME = "qq_lora_crop_resolution_c1000_seed42_v1"
CONFIG = "configs/qq_lora_ablation.yaml"
MINUTES_PER_ARM = 12
NUM_CLASSES_SUBSET = 1000
SEED = 42
NUM_WORKERS = 8
DOWNLOAD_DATA = True
RUN_OPTIONAL_448 = False  # Enable only after reviewing A–D.
# Keep the same name/settings to reuse completed arms. Change the name for a
# new seed or recipe. Interrupted arms are not silently resumed from partial weights.


## 1. Connect to GitHub
Same hidden-token connection as your training launcher. Checkout/pull `qq_lora_ablation`; this does not merge it with main.


In [ ]:
import sys
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


## 2. Install packages and check the GPU


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime; A100 is recommended."
print("GPU:", torch.cuda.get_device_name())
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if 'A100' not in torch.cuda.get_device_name():
    print("For comparable timing, run all arms on the same A100. Do not mix hardware between arms.")


## 3. Mount Drive
Use the same `CS604_RUNS` shared-folder shortcut as before. Dataset images stay on temporary Colab storage; results and checkpoint files go to Drive.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
base_out = Path(DRIVE_OUT)
if not base_out.is_dir():
    raise FileNotFoundError(f"{DRIVE_OUT} not found. Add the shared CS604_RUNS folder as a shortcut in My Drive.")
EXPERIMENT_DIR = base_out / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment:", EXPERIMENT_DIR)


## 4. Download or reuse extracted images
The full download/extraction can be slow and needs room for archives plus images. Check disk space first. Set `DOWNLOAD_DATA=False` to use an existing extract. Reuse is based on a small path check; it does not certify every file.


In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
        process.wait()
        raise


import shutil
root = Path(DATA_DIR)
root.mkdir(parents=True, exist_ok=True)
print(f"Free Colab disk: {shutil.disk_usage(root).free / 2**30:.1f} GiB")
def extracted_training_available():
    listing = root / 'train.txt'
    if not listing.is_file(): return False
    paths = []
    with listing.open() as handle:
        for line in handle:
            if line.strip(): paths.append(line.split()[0])
            if len(paths) == 20: break
    return bool(paths) and all((root / path).is_file() for path in paths)
if DOWNLOAD_DATA and not extracted_training_available():
    run_live([sys.executable, '-u', 'scripts/download_data.py', '--dest', DATA_DIR])
else:
    print('Using existing dataset:', root)
assert (root / 'train.txt').is_file(), 'Set DATA_DIR to the extracted training dataset.'


## 5. Review settings before training
Each arm starts from the original pretrained backbone, not another arm's checkpoint. A/B share 336 fit-only prototypes; C/D share separately built 384 prototypes. Evaluation uses the same **center crop 0.9**, without flip or zoom. We compare **final accuracy after equal training minutes**; best-epoch accuracy is recorded for reference.

Both crop branches still receive the existing flip and color jitter. The padded branch resizes the whole image without stretching, then adds borders filled with the model's mean color. Full-image views can preserve complete subjects but also make small ones smaller.

This subset uses a smaller fraction of the available classes, so its scores are not comparable to full-data CodaBench results. The shared trainer uses a 5% time warmup and cosine decay to the end of each arm's budget, with no epoch ceiling.


In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)
print("Backbone:", cfg.model_name)
print("LoRA rank:", cfg.lora_rank, "targets:", cfg.lora_targets)
print("Extra trainable parts:", cfg.unfreeze_extra)
print("Classifier:", cfg.head_type, "initialized from fit-only prototypes")
print("LoRA / head LR:", cfg.lr, cfg.head_lr)
print("Batch / accumulation:", cfg.batch_size, cfg.grad_accum_steps)
print("Fitting classes:", NUM_CLASSES_SUBSET, "holdout/class:", cfg.ablation_holdout)
print("Minutes per arm:", MINUTES_PER_ARM, "A–D total:", MINUTES_PER_ARM * 4)
print("Optional E adds:", MINUTES_PER_ARM, "training minutes")
print("Additional wall time is needed for prototypes, evaluation, loading weights and saving.")
assert 0 < MINUTES_PER_ARM <= 144
assert cfg.model_name == 'vit_base_patch16_224.augreg2_in21k_ft_in1k'
assert cfg.lora_rank == 24 and cfg.prompt_length == 0


## 6. Run A–D
This runs sequentially and saves each completed arm immediately. Completed arms are reused only if settings, fitting/holdout lists, code, package versions and saved checkpoints match. If an arm was interrupted, use a new experiment name rather than silently mixing partial and full runs. Keep the same Colab runtime/hardware for a fair speed comparison.


In [ ]:
command = [sys.executable, '-u', '-m', 'src.qq_lora_ablation',
           '--config', CONFIG, '--data_root', DATA_DIR, '--output_dir', str(EXPERIMENT_DIR),
           '--minutes', str(MINUTES_PER_ARM), '--classes', str(NUM_CLASSES_SUBSET),
           '--seed', str(SEED), '--workers', str(NUM_WORKERS)]
run_live(command + ['--stage', 'main'])


## 7. Review the four results
Accuracy is shown as a percentage; speed includes training-time image loading. Compare B−A and D−C for crop effects, and C−A and D−B for resolution effects. A result from one seed can be noisy; repeat promising comparisons with another seed before a full-data run.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(EXPERIMENT_DIR / 'results.csv')
display(results[['arm','resolution','full_image_probability','final_val_acc','best_val_acc',
                 'training_minutes','epochs','images_per_second','peak_vram_gib','adaptation_pct']])
main_results = results[results.arm.isin(['A','B','C','D'])].set_index('arm')
for label,left,right in [('Padding at 336','B','A'),('Padding at 384','D','C'),
                         ('Resolution with crops','C','A'),('Resolution with mixed views','D','B')]:
    delta = 100*(main_results.loc[left,'final_val_acc']-main_results.loc[right,'final_val_acc'])
    print(f'{label}: {delta:+.2f} percentage points')
fig,axes = plt.subplots(1,3,figsize=(15,4))
for ax,field,title in zip(axes,['final_val_acc','images_per_second','peak_vram_gib'],['Final holdout accuracy (%)','Training images / second','Peak GPU memory (GiB)']):
    values = results[field]*100 if field=='final_val_acc' else results[field]
    ax.bar(results.arm,values); ax.set_title(title)
fig.tight_layout(); fig.savefig(EXPERIMENT_DIR/'comparison.png',dpi=150); plt.show()
selection = json.loads((EXPERIMENT_DIR/'selection.json').read_text())
if selection['candidate_384']:
    print('384 candidate for optional 448 comparison:', selection['candidate_384']['arm'])
else:
    print('384 has not beaten the best 336 score by more than 0.3 percentage points. Keep 448 off for now.')


## 8. Optional: compare the better 384 setting with 448
Set `RUN_OPTIONAL_448=True` in Settings, rerun that settings cell, then run this cell. **It adds one fresh 448 arm; it reuses the completed 384 arm.** The crop choice, fitting/holdout images, optimizer and training minutes match. Prototypes are freshly built at 448 from fitting images only.

To avoid spending time on a tiny apparent gain, this runs only if the best 384 arm exceeds the best 336 arm by more than **0.3 percentage points**. This is a screening rule, not a statistical guarantee. No 448 run starts if A–D are missing. Rerunning the results cell above includes E.


In [ ]:
if RUN_OPTIONAL_448:
    run_live(command + ['--stage', '448'])
    extended = pd.read_csv(EXPERIMENT_DIR/'results.csv')
    if 'E_448' in extended.arm.values:
        selected = json.loads((EXPERIMENT_DIR/'selection.json').read_text())['candidate_384']
        comparison = extended[extended.arm.isin([selected['arm'],'E_448'])]
        display(comparison[['arm','resolution','full_image_probability','final_val_acc',
                            'training_minutes','epochs','images_per_second','peak_vram_gib']])
        e = extended[extended.arm.eq('E_448')].iloc[0]
        print(f"448 minus matched 384: {100*(e.final_val_acc-selected['final_val_acc']):+.2f} percentage points")
else:
    print('Optional 448 test is off. Review A–D first, then enable RUN_OPTIONAL_448 if appropriate.')


## 9. Training curves and what to do next
Each arm folder contains its resolved config, best/last checkpoints and normal training logs. `results.csv` records final/best accuracy, training minutes, wall minutes, completed epochs, speed, peak memory and the backbone adaptation percentage (classifier excluded). Fitting/holdout lists are copied into `split_manifests/` on Drive; image links stay on temporary Colab disk.

Choose a promising setting only if the gain is worth its speed and memory cost. Repeat close comparisons with another seed, using a new experiment name. A final full-data run would need **fresh all-data prototypes and pretrained weights**, all 450,000 images and a separate 144-minute budget. This notebook does not automatically launch that run or generate official submissions. Model-guided zoom can be tested afterward.


In [ ]:
for row in pd.read_csv(EXPERIMENT_DIR/'results.csv').itertuples(index=False):
    run_live([sys.executable,'scripts/plot_curves.py','--run_dir',row.run_dir,
              '--out_dir',str(EXPERIMENT_DIR / row.arm / 'plots')])
print('Saved results:', EXPERIMENT_DIR)
